### Паспорт объёмов, сравнение режимов и структура источников урона по башням

Проверяем физические размеры созданных Iceberg-таблиц и базовые распределения по витринам `MART`.

**Ключевые результаты:**
* **Сжатие данных (`STAGE` → `MART`):**
  * `stage.case13_match_players`: 3 файла, 48,20 МБ, 4 845 102 строки.
  * `stage.case13_tower_damage`: 120 файлов, 2 392,29 МБ (2,39 ГБ), 246 912 008 строк.
  * `mart.case13_match_summary`: 8 файлов, 120,82 МБ, 3 216 290 строк (только согласованные матчи).
  * `mart.case13_tower_ability`: 16 файлов, 291,09 МБ, 18 080 385 строк (сжатие числа строк в 13,7 раза и объёма в 8,2 раза относительно `STAGE` событий).
* **Разделение `Standard` vs `Turbo`:** в `Standard` (217 331 матч) средняя длительность 40,6 мин (медиана 39,8 мин), в `Turbo` (103 382 матча) — 25,4 мин (медиана 24,3 мин). Средний урон игрока умениями и саммонами по башням — 281,0 против 210,9.
* **Источники урона по башням (`Standard`):** `auto_attack` - 84,13% (3,55 млрд), `summon_without_ability` — 6,75% (284,90 млн), `ability_direct` — 5,97% (252,26 млн), `ability_summon` - 1,74% (73,50 млн), `item` - 1,41% (59,56 млн). Итого на способности и призыв приходится **14,46%** урона по башням.
* **Топ источников по валовому урону (`Standard`):** `npc_dota_invoker_forged_spirit` (70,25 млн, 35 342 матча), `Bone Guard` (54,97 млн), `Moon Glaives` (41,86 млн), `npc_dota_furion_treant` (37,30 млн), `Elder Dragon Form` (34,21 млн). 7 из 15 лидеров относятся к `summon_without_ability`.

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window


with SparkSessionContext('test-spark-sql') as spark:
    
    mart_summary = spark.table("lakehouse.mart.case13_match_summary")
    mart_detail  = spark.table("lakehouse.mart.case13_tower_ability")
    
    print("=== 1. Паспорт объёмов: STAGE vs MART ===")
    for tbl in [
        "lakehouse.stage.case13_match_players",
        "lakehouse.stage.case13_tower_damage",
        "lakehouse.mart.case13_match_summary",
        "lakehouse.mart.case13_tower_ability"
    ]:
        spark.sql(f"""
            SELECT
                '{tbl}' AS table_name,
                COUNT(*) AS files_cnt,
                ROUND(SUM(file_size_in_bytes) / 1024.0 / 1024.0, 2) AS size_mb,
                SUM(record_count) AS rows_cnt
            FROM {tbl}.files
        """).show(truncate=False)

    print("=== 3.1. Сравнение режимов Standard vs Turbo (по case13_match_summary) ===")
    mart_summary.filter(F.col("match_has_tde_events") == True).groupBy("game_mode_group").agg(
        F.countDistinct("match_id").alias("matches"),
        F.round(F.avg("duration") / 60.0, 1).alias("avg_duration_min"),
        F.round(F.expr("percentile_approx(duration, 0.5)") / 60.0, 1).alias("median_duration_min"),
        F.round(F.avg("ev_towers_damage"), 1).alias("avg_player_tower_dmg"),
        F.round(F.avg(F.col("ev_towers_ability_damage") + F.col("ev_towers_summon_damage")), 1).alias("avg_player_ab_sum_dmg")
    ).show(truncate=False)
    
    print("=== 3.2. Структура источников урона ПО БАШНЯМ (Standard режим) ===")
    w_std = Window.partitionBy()
    mart_detail.filter(
        (F.col("building_category") == "TOWER") & (F.col("game_mode_group") == "Standard")
    ).groupBy("damage_source_type").agg(
        F.sum("events_count").alias("events"),
        F.sum("total_damage").alias("total_dmg"),
        F.countDistinct("match_id").alias("matches")
    ).withColumn("dmg_share_pct", F.round(F.col("total_dmg") * 100.0 / F.sum("total_dmg").over(w_std), 2)) \
     .orderBy(F.desc("total_dmg")).show(truncate=False)
    
    print("=== 3.3. ТОП-15 способностей и саммонов по урону по башням (Standard, без автоатак) ===")
    mart_detail.filter(
        (F.col("building_category") == "TOWER") &
        (F.col("game_mode_group") == "Standard") &
        (F.col("is_ability_or_summon") == True)
    ).groupBy("source_label", "damage_source_type", "hero_name").agg(
        F.countDistinct("match_id").alias("matches_with_dmg"),
        F.sum("total_damage").alias("total_dmg"),
        F.round(F.sum("total_damage") * 1.0 / F.countDistinct("match_id"), 1).alias("avg_dmg_per_active_match")
    ).orderBy(F.desc("total_dmg")).show(15, truncate=False)

=== 1. Паспорт объёмов: STAGE vs MART ===


+------------------------------------+---------+-------+--------+
|table_name                          |files_cnt|size_mb|rows_cnt|
+------------------------------------+---------+-------+--------+
|lakehouse.stage.case13_match_players|3        |48.20  |4845102 |
+------------------------------------+---------+-------+--------+

+-----------------------------------+---------+-------+---------+
|table_name                         |files_cnt|size_mb|rows_cnt |
+-----------------------------------+---------+-------+---------+
|lakehouse.stage.case13_tower_damage|120      |2392.29|246912008|
+-----------------------------------+---------+-------+---------+

+-----------------------------------+---------+-------+--------+
|table_name                         |files_cnt|size_mb|rows_cnt|
+-----------------------------------+---------+-------+--------+
|lakehouse.mart.case13_match_summary|8        |120.82 |3216290 |
+-----------------------------------+---------+-------+--------+

+-----------

+---------------+-------+----------------+-------------------+--------------------+---------------------+
|game_mode_group|matches|avg_duration_min|median_duration_min|avg_player_tower_dmg|avg_player_ab_sum_dmg|
+---------------+-------+----------------+-------------------+--------------------+---------------------+
|Turbo          |103382 |25.4            |24.3               |1590.5              |210.9                |
|Standard       |217331 |40.6            |39.8               |1943.3              |281.0                |
+---------------+-------+----------------+-------------------+--------------------+---------------------+

=== 3.2. Структура источников урона ПО БАШНЯМ (Standard режим) ===


26/11/14 10:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 10:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 10:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 10:54:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 10:54:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 10:54:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/14 1

+----------------------+--------+----------+-------+-------------+
|damage_source_type    |events  |total_dmg |matches|dmg_share_pct|
+----------------------+--------+----------+-------+-------------+
|auto_attack           |85580443|3553207378|217325 |84.13        |
|summon_without_ability|16751942|284903647 |111034 |6.75         |
|ability_direct        |10584339|252260525 |137515 |5.97         |
|ability_summon        |6854653 |73496592  |36387  |1.74         |
|item                  |960175  |59555218  |32690  |1.41         |
+----------------------+--------+----------+-------+-------------+

=== 3.3. ТОП-15 способностей и саммонов по урону по башням (Standard, без автоатак) ===


+------------------------------+----------------------+----------------+----------------+---------+------------------------+
|source_label                  |damage_source_type    |hero_name       |matches_with_dmg|total_dmg|avg_dmg_per_active_match|
+------------------------------+----------------------+----------------+----------------+---------+------------------------+
|npc_dota_invoker_forged_spirit|summon_without_ability|Invoker         |35342           |70248035 |1987.7                  |
|Bone Guard                    |ability_summon        |Wraith King     |25514           |54973798 |2154.7                  |
|Moon Glaives                  |ability_direct        |Luna            |25767           |41855926 |1624.4                  |
|npc_dota_furion_treant        |summon_without_ability|Nature's Prophet|19319           |37301297 |1930.8                  |
|Elder Dragon Form             |ability_direct        |Dragon Knight   |26127           |34209966 |1309.4                  |


### Проверка гипотез H1 и H2

Выполняем проверку основных гипотез в `Standard`-режиме (217 331 матч с событиями):
* **H1:** Топ-15 героев по удельному урону умениями/саммонами по башням (порог >= 500 сыгранных матчей, с учётом 0-матчей, медианы и темпа в минуту).
* **H2:** Матрица доли умений и саммонов по срезу `phase × tower_type` (`T1–T4`) и сравнение лидеров ранней (`0-10, T1`) и поздней (`30+, T3/T4`) стадий.
* **Дополнительно:** Сравнение `win = true` vs `win = false` по корзинам длительности в полных командах (5 игроков).

**Ключевые результаты:**
* **Гипотеза H1 подтверждена:** после нормализации на сыгранный матч и на минуту игры лидируют специализированные саммонеры и пушеры:
  * `Lone Druid`: среднее 4 030,1 (медиана 2 777, темп 98,45/мин, доля в башенном уроне героя 85,2%, 6 195 матчей).
  * `Visage`: среднее 3 370,8 (медиана 2 505, темп 92,44/мин).
  * `Pugna`: среднее 3 097,2 (медиана 2 474, темп 81,61/мин, активен в 90,2% игр).
  * `Leshrac`: среднее 3 035,4 (медиана 2 102, темп 80,26/мин).
  * `Lycan`: среднее 3 001,5 (медиана 2 244, темп 84,46/мин).
  * Популярный в валовом топе `Invoker` (40 491 матч) имеет почти вдвое меньший удельный вклад: среднее 1 735,5 (медиана 1 318, темп 42,92/мин).
* **Гипотеза H2 подтверждена:**
  * Доля умений и саммонов последовательно падает от ранней игры к лейту внутри каждого тира башен: для `T1` с 23,33% (`0-10`) до 10,73% (`30+`), для `T2` с 32,96% до 10,35%, для `T3` с 26,63% до 11,18%. На `T4` в фазе `30+` доля выше (13,77%) за счёт лейт-механик вроде `Moon Glaives`.
  * На ранней стадии (`0-10, T1`) лидируют `Bone Guard` (4,55 млн), `Elder Dragon Form` (3,45 млн), `Nether Blast` (3,37 млн); на поздней (`30+, T3/T4`) — `Moon Glaives` (27,10 млн), `Forged Spirit` (20,29 млн), `Shadow Shaman Ward 3` (18,12 млн).
* **Сравнение победителей и проигравших (5v5):** во всех корзинах длительности у победителей кратно выше и общий башенный урон, и темп урона умениями/саммонами в минуту (например, в 20-30m: 18,77/мин против 1,51/мин; в 40+m: 8,63/мин против 4,06/мин).

In [ ]:
from lakehouse_spark import SparkSessionContext
from pyspark.sql import functions as F
from pyspark.sql import Window


with SparkSessionContext('t2') as spark:

    # Топ-15 героев по нормализованному урону умений+саммонов по башням (Standard)
    mart_summary = spark.table("lakehouse.mart.case13_match_summary")
    mart_detail  = spark.table("lakehouse.mart.case13_tower_ability")
    
    print("=== 4.1. [H1] Топ-15 героев по вкладу способностей и саммонов (с учётом 0-матчей, Standard) ===")
    ms_std = mart_summary.filter(
        (F.col("game_mode_group") == "Standard") & (F.col("match_has_tde_events") == True)
    ).withColumn(
        "ab_sum_dmg", F.col("ev_towers_ability_damage") + F.col("ev_towers_summon_damage")
    ).withColumn(
        "ab_sum_per_min",
        F.when(F.col("duration") > 0, F.col("ab_sum_dmg") / (F.col("duration") / 60.0)).otherwise(0.0)
    ).withColumn(
        "towers_per_min",
        F.when(F.col("duration") > 0, F.col("ev_towers_damage") / (F.col("duration") / 60.0)).otherwise(0.0)
    )
    
    ms_std.groupBy("hero_name", "is_pusher_role").agg(
        F.count("*").alias("played_matches"),
        F.sum(F.when(F.col("ab_sum_dmg") > 0, 1).otherwise(0)).alias("matches_with_ab_dmg"),
        F.round(F.avg("ev_towers_damage"), 1).alias("avg_towers_dmg_all"),
        F.round(F.avg("ab_sum_dmg"), 1).alias("avg_ab_sum_dmg_all"),
        F.round(F.avg("ab_sum_per_min"), 2).alias("avg_ab_sum_per_min"),
        F.round(F.expr("percentile_approx(ab_sum_dmg, 0.5)"), 1).alias("median_ab_sum_all"),
        F.round(F.sum("ab_sum_dmg") * 100.0 / F.sum("ev_towers_damage"), 1).alias("ab_sum_share_of_hero_tower_pct")
    ).filter(F.col("played_matches") >= 500) \
     .withColumn("pct_matches_active", F.round(F.col("matches_with_ab_dmg") * 100.0 / F.col("played_matches"), 1)) \
     .orderBy(F.desc("avg_ab_sum_dmg_all")) \
     .show(15, truncate=False)
    
    # Изменение доли способностей и саммонов по фазам и тирам башен (Standard)
    print("=== 4.2. [H2] Структура урона по башням: Фаза x Тип башни (Standard) ===")
    md_std_towers = mart_detail.filter(
        (F.col("building_category") == "TOWER") & (F.col("game_mode_group") == "Standard")
    )
    
    md_std_towers.groupBy("phase", "tower_type").agg(
        F.countDistinct("match_id").alias("matches"),
        F.sum("total_damage").alias("total_dmg"),
        F.sum(F.when(F.col("is_ability_or_summon"), F.col("total_damage")).otherwise(0)).alias("ab_sum_dmg")
    ).withColumn(
        "ab_sum_share_pct", F.round(F.col("ab_sum_dmg") * 100.0 / F.col("total_dmg"), 2)
    ).orderBy("phase", "tower_type").show(20, truncate=False)
    
    print("=== 4.3. [H2] Топ-5 способностей/саммонов на ранней фазе (0-10, T1) vs поздней (30+, T3/T4) ===")
    print("--- Ранняя фаза: 0-10 мин, T1 ---")
    md_std_towers.filter((F.col("phase") == "0-10") & (F.col("tower_type") == "T1") & (F.col("is_ability_or_summon") == True)) \
        .groupBy("source_label", "hero_name") \
        .agg(F.sum("total_damage").alias("dmg"), F.countDistinct("match_id").alias("matches")) \
        .orderBy(F.desc("dmg")).show(5, truncate=False)
    
    print("--- Поздняя фаза: 30+ мин, T3/T4 ---")
    md_std_towers.filter((F.col("phase") == "30+") & (F.col("tower_type").isin("T3", "T4")) & (F.col("is_ability_or_summon") == True)) \
        .groupBy("source_label", "hero_name") \
        .agg(F.sum("total_damage").alias("dmg"), F.countDistinct("match_id").alias("matches")) \
        .orderBy(F.desc("dmg")).show(5, truncate=False)
    
    # Сравнение победителей и проигравших по длительности (Standard, полные команды 5v5)
    print("=== 4.4. [H3] Победители vs Проигравшие с контролем длительности (Standard, 5 игроков) ===")
    ms_std.filter(F.col("team_players_count") == 5).groupBy("duration_bucket", "win").agg(
        F.count("*").alias("player_matches"),
        F.round(F.avg("ev_towers_damage"), 1).alias("avg_towers_dmg"),
        F.round(F.expr("percentile_approx(ev_towers_damage, 0.5)"), 1).alias("median_towers_dmg"),
        F.round(F.avg("ab_sum_dmg"), 1).alias("avg_ab_sum_dmg"),
        F.round(F.avg("ab_sum_per_min"), 2).alias("avg_ab_sum_per_min"),
        F.round(F.sum("ab_sum_dmg") * 100.0 / F.sum("ev_towers_damage"), 2).alias("ab_sum_share_pct")
    ).orderBy("duration_bucket", "win").show(20, truncate=False)

=== 4.1. [H1] Топ-15 героев по вкладу способностей и саммонов (с учётом 0-матчей, Standard) ===


+----------------+--------------+--------------+-------------------+------------------+------------------+------------------+-----------------+------------------------------+------------------+
|hero_name       |is_pusher_role|played_matches|matches_with_ab_dmg|avg_towers_dmg_all|avg_ab_sum_dmg_all|avg_ab_sum_per_min|median_ab_sum_all|ab_sum_share_of_hero_tower_pct|pct_matches_active|
+----------------+--------------+--------------+-------------------+------------------+------------------+------------------+-----------------+------------------------------+------------------+
|Lone Druid      |true          |6195          |5285               |4727.8            |4030.1            |98.45             |2777             |85.2                          |85.3              |
|Visage          |true          |3659          |3112               |4611.3            |3370.8            |92.44             |2505             |73.1                          |85.1              |
|Pugna           |true        

+-----+----------+-------+---------+----------+----------------+
|phase|tower_type|matches|total_dmg|ab_sum_dmg|ab_sum_share_pct|
+-----+----------+-------+---------+----------+----------------+
|0-10 |T1        |201616 |141124961|32927754  |23.33           |
|0-10 |T2        |7664   |4232211  |1395041   |32.96           |
|0-10 |T3        |439    |529153   |140899    |26.63           |
|0-10 |T4        |46     |75971    |11565     |15.22           |
|10-20|T1        |215459 |467131444|87750836  |18.79           |
|10-20|T2        |131842 |189872055|45914807  |24.18           |
|10-20|T3        |16220  |24625446 |5395778   |21.91           |
|10-20|T4        |3176   |8504509  |1106605   |13.01           |
|20-30|T1        |152131 |134487480|20541943  |15.27           |
|20-30|T2        |200445 |599348526|88869970  |14.83           |
|20-30|T3        |114574 |338990041|50117741  |14.78           |
|20-30|T4        |39447  |100234985|17032780  |16.99           |
|30+  |T1        |52586  

+------------------------------+----------------+-------+-------+
|source_label                  |hero_name       |dmg    |matches|
+------------------------------+----------------+-------+-------+
|Bone Guard                    |Wraith King     |4549578|16396  |
|Elder Dragon Form             |Dragon Knight   |3453682|10671  |
|Nether Blast                  |Pugna           |3367431|6458   |
|npc_dota_invoker_forged_spirit|Invoker         |2444790|11619  |
|npc_dota_furion_treant        |Nature's Prophet|2213337|6448   |
+------------------------------+----------------+-------+-------+
only showing top 5 rows
--- Поздняя фаза: 30+ мин, T3/T4 ---


+------------------------------+-------------+--------+-------+
|source_label                  |hero_name    |dmg     |matches|
+------------------------------+-------------+--------+-------+
|Moon Glaives                  |Luna         |27102781|14147  |
|npc_dota_invoker_forged_spirit|Invoker      |20289459|19845  |
|npc_dota_shadow_shaman_ward_3 |Shadow Shaman|18122949|12835  |
|npc_dota_lone_druid_bear1     |Lone Druid   |11938056|2983   |
|Elder Dragon Form             |Dragon Knight|10550832|14670  |
+------------------------------+-------------+--------+-------+
only showing top 5 rows
=== 4.4. [H3] Победители vs Проигравшие с контролем длительности (Standard, 5 игроков) ===


+---------------+-----+--------------+--------------+-----------------+--------------+------------------+----------------+
|duration_bucket|win  |player_matches|avg_towers_dmg|median_towers_dmg|avg_ab_sum_dmg|avg_ab_sum_per_min|ab_sum_share_pct|
+---------------+-----+--------------+--------------+-----------------+--------------+------------------+----------------+
|20-30m         |false|109835        |164.3         |0                |41.1          |1.51              |25.0            |
|20-30m         |true |109835        |3140.3        |2026             |498.9         |18.77             |15.89           |
|30-40m         |false|432655        |404.6         |83               |86.9          |2.44              |21.49           |
|30-40m         |true |432655        |3131.5        |1898             |430.2         |12.32             |13.74           |
|40+m           |false|533560        |1216.9        |465              |199.2         |4.06              |16.37           |
|40+m           